# Spin-chain time evolution: scipy.expm vs Trotter

Time-evolve a 3-spin TFI chain by t=0.1. Same `expv(H, ψ, t)` trace runs scipy.expm on cpu and a Trotter circuit on the simulator route.

**Hardware-agnostic by design.** This notebook traces ONE computation with the uniqx SDK; the engine picks the route (classical / simulator / accelerator) at preflight time.

## Background

**Problem:** `e^{-iHt} · ψ`. **Classical:** `scipy.linalg.expm`. **Quantum:** Trotter / qDRIFT / LCU. Both solve the same time-evolution problem.

## Setup

In [ ]:
import os
import numpy as np
from uniqx.core.execution import connect, preflight, submit, get
from uniqx.core.tracing import trace
from uniqx import login

GATEWAY_ADDR = os.environ.get("GATEWAY_ADDR", "api.oriqx.com:443")
login(os.environ["UNIQX_API_KEY"], gateway=GATEWAY_ADDR)
client = connect(GATEWAY_ADDR)


## Trace the computation

In [ ]:
from uniqx.core import types
from uniqx.core.enums import SpinChainModel
from uniqx.domains.physics.kernels import spin_chain_hamiltonian
from uniqx.ops import shape
from uniqx.ops.primitives.evolution import expv

n_spins = 3
dim = 1 << n_spins
t = 0.1

@trace
def prog(psi_0):
    h_flat = spin_chain_hamiltonian(n_spins=n_spins, model=SpinChainModel.TFI,
                                     J=1.0, h_field=0.5, pbc=False)
    h_mat = shape.reshape(h_flat, result_type=types.tensor("f64", [dim, dim]),
                           shape=[dim, dim])
    return expv(h_mat, psi_0, t, A_is_generator=False, hermitian=True)

psi_init = np.zeros(dim); psi_init[0] = 1.0
module = prog(psi_init.tolist())


## Preflight — see what routes the engine found

uniqx is hardware-agnostic: the same trace runs on every available route. `preflight` returns the engine's choices.

In [ ]:
options = preflight(module, client=client)
print("Available routes:")
for o in options:
    print(f"  {o['label']:25s}  score={o.get('score', 'n/a')}")


## Run on every available route

Production usage is just `client.run(module)` — the engine picks the best route automatically. Here we run on every route for side-by-side comparison.

In [ ]:
runs = {}
for o in options:
    label = o.get("label") or ""
    print(f"\n--- Running on {label} ---")
    job_id = submit(module, client=client, preflight_job_id=options.job_id, option_idx=o['_idx'])
    res = get(job_id, client=client, wait=True, timeout=120)
    runs[label] = res
    print(f"  done: {str(res)[:200]}")
